# Model 1: Optimized MLP with Hyperparameter Tuning
This notebook builds upon Assignment 1 by applying advanced optimization techniques
and hyperparameter tuning to a Multi-Layer Perceptron (MLP) classifier.

## Key Improvements over Assignment 1:
- Larger feature set (TF-IDF + VADER + TextBlob)
- Multiple optimizers compared (SGD, Adam, RMSProp)
- Batch Normalization
- Dropout regularization
- Learning rate scheduling
- Hyperparameter tuning with GridSearchCV

In [ ]:
# ─────────────────────────────────────────────
# CELL 1: Install dependencies
# ─────────────────────────────────────────────
!pip install nltk textblob vaderSentiment scikit-learn tensorflow kagglehub -q

In [ ]:
# ─────────────────────────────────────────────
# CELL 2: Import all required libraries
# ─────────────────────────────────────────────
import kagglehub
import pandas as pd
import numpy as np
import re
import nltk
import json
import matplotlib.pyplot as plt
import seaborn as sns

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

# Scikit-learn tools for classical ML pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)

# TensorFlow / Keras for deep MLP with full optimizer control
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("TensorFlow version:", tf.__version__)
nltk.download('vader_lexicon')

In [ ]:
# ─────────────────────────────────────────────
# CELL 3: Load the IMDB dataset from Kaggle
# ─────────────────────────────────────────────
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
data = pd.read_csv(f"{path}/IMDB Dataset.csv")

# Remove duplicate rows (418 found in Assignment 1 EDA)
data = data.drop_duplicates(subset='review').reset_index(drop=True)
print(f"Dataset shape after dedup: {data.shape}")
data.head()

In [ ]:
# ─────────────────────────────────────────────
# CELL 4: Text Preprocessing
# More thorough than Assignment 1 — also removes
# punctuation and stopwords for TF-IDF features
# ─────────────────────────────────────────────
nltk.download('stopwords')
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words('english'))

def clean_text_basic(text):
    """Basic cleaning used for sentiment features (preserves sentiment words)."""
    text = text.lower()                          # lowercase
    text = re.sub(r"<.*?>", "", text)            # remove HTML tags
    text = re.sub(r"\s+", " ", text)             # collapse whitespace
    return text.strip()

def clean_text_deep(text):
    """Deep cleaning for TF-IDF: removes punctuation & stopwords."""
    text = clean_text_basic(text)
    text = re.sub(r"[^a-z\s]", "", text)         # keep only letters
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS]  # remove stopwords
    return " ".join(tokens)

# Apply both cleaning functions
data['clean_basic'] = data['review'].apply(clean_text_basic)   # for sentiment features
data['clean_deep']  = data['review'].apply(clean_text_deep)    # for TF-IDF

# Encode labels: positive=1, negative=0
data['label'] = data['sentiment'].map({'positive': 1, 'negative': 0})

print("Sample cleaned review (basic):")
print(data['clean_basic'][0][:200])

In [ ]:
# ─────────────────────────────────────────────
# CELL 5: Feature Engineering
# Assignment 1 used only 5 sentiment features.
# Here we combine:
#   (a) VADER + TextBlob scores (5 features)
#   (b) TF-IDF unigrams + bigrams (5000 features)
# This richer representation should boost accuracy.
# ─────────────────────────────────────────────
from scipy.sparse import hstack, csr_matrix

# ── 5a: VADER + TextBlob sentiment features ──
vader = SentimentIntensityAnalyzer()

def extract_sentiment_features(text):
    """
    Returns a 5-dimensional feature vector:
      [vader_neg, vader_neu, vader_pos, vader_compound, textblob_polarity]
    """
    vs = vader.polarity_scores(text)
    tb = TextBlob(text).sentiment.polarity
    return [vs['neg'], vs['neu'], vs['pos'], vs['compound'], tb]

print("Extracting sentiment features (may take ~1 min)...")
sent_features = np.array([extract_sentiment_features(t) for t in data['clean_basic']])
print(f"Sentiment feature matrix shape: {sent_features.shape}")

# ── 5b: TF-IDF features ──
# max_features=5000 keeps vocabulary manageable
# ngram_range=(1,2) captures both unigrams and bigrams
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=5)
tfidf_features = tfidf.fit_transform(data['clean_deep'])   # sparse matrix
print(f"TF-IDF feature matrix shape: {tfidf_features.shape}")

# ── Combine both feature sets ──
# Convert sentiment features to sparse so hstack works
X_combined = hstack([tfidf_features, csr_matrix(sent_features)]).toarray()
y = data['label'].values
print(f"Combined feature matrix shape: {X_combined.shape}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 6: Train / Validation / Test Split
# Same 60/20/20 split as Assignment 1 for fair comparison
# stratify=y ensures balanced classes in each split
# ─────────────────────────────────────────────
X_temp, X_test, y_temp, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"Train:      {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test:       {X_test.shape}")

# ── Feature Scaling ──
# StandardScaler normalizes each feature to zero mean & unit variance.
# This is critical for gradient-based optimizers to converge quickly.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit ONLY on train
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

In [ ]:
# ─────────────────────────────────────────────
# CELL 7: Build a Keras MLP with:
#   • Batch Normalization after each hidden layer
#   • Dropout regularization to prevent overfitting
#   • He initialization (best for ReLU)
# ─────────────────────────────────────────────
def build_mlp(input_dim, hidden_sizes=(256, 128, 64),
              dropout_rate=0.3, learning_rate=1e-3,
              optimizer_name='adam'):
    """
    Builds a Keras Sequential MLP.

    Args:
        input_dim     : number of input features
        hidden_sizes  : tuple defining neurons in each hidden layer
        dropout_rate  : fraction of neurons to drop during training
        learning_rate : step size for weight updates
        optimizer_name: 'adam' | 'sgd' | 'rmsprop'

    Returns:
        compiled Keras model
    """
    model = keras.Sequential(name='MLP_Optimized')
    model.add(layers.Input(shape=(input_dim,)))

    for i, units in enumerate(hidden_sizes):
        # Dense layer with He-normal initialization (optimal for ReLU)
        model.add(layers.Dense(
            units,
            kernel_initializer='he_normal',
            name=f'dense_{i+1}'
        ))
        # Batch Normalization: normalizes activations → faster + more stable training
        model.add(layers.BatchNormalization(name=f'bn_{i+1}'))
        # ReLU activation
        model.add(layers.Activation('relu', name=f'relu_{i+1}'))
        # Dropout: randomly zeros fraction of neurons to reduce overfitting
        model.add(layers.Dropout(dropout_rate, name=f'dropout_{i+1}'))

    # Output layer: 1 neuron with sigmoid → outputs P(positive)
    model.add(layers.Dense(1, activation='sigmoid', name='output'))

    # ── Select optimizer ──
    # Adam:    adaptive lr per parameter, momentum + RMSProp combined → best default
    # SGD+mom: classical momentum; good when well-tuned
    # RMSProp: adaptive lr, good for non-stationary problems
    if optimizer_name == 'adam':
        opt = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == 'sgd':
        # Momentum SGD: uses EWMA of past gradients to smooth updates
        opt = keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9, nesterov=True)
    elif optimizer_name == 'rmsprop':
        opt = keras.optimizers.RMSprop(learning_rate=learning_rate)
    else:
        raise ValueError(f"Unknown optimizer: {optimizer_name}")

    # Binary cross-entropy loss for binary classification
    model.compile(
        optimizer=opt,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Build with default Adam optimizer
model = build_mlp(input_dim=X_train_s.shape[1])
model.summary()

In [ ]:
# ─────────────────────────────────────────────
# CELL 8: Train the model with callbacks
#
# Callbacks used:
#   • EarlyStopping   – stops when val_loss stops improving
#   • ReduceLROnPlateau – halves lr when plateau detected
# ─────────────────────────────────────────────
callbacks = [
    # Stop training when validation loss doesn't improve for 5 epochs
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    # Reduce learning rate by factor 0.5 when val_loss plateaus for 3 epochs
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1, min_lr=1e-6)
]

history = model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=50,           # max epochs (early stopping will halt earlier)
    batch_size=256,      # mini-batch gradient descent
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# ─────────────────────────────────────────────
# CELL 9: Compare Optimizers
# Train same architecture with Adam, SGD, RMSProp
# and compare validation accuracy
# ─────────────────────────────────────────────
optimizer_results = {}

for opt_name in ['adam', 'sgd', 'rmsprop']:
    print(f"\n── Training with {opt_name.upper()} ──")
    m = build_mlp(input_dim=X_train_s.shape[1], optimizer_name=opt_name)
    cb = [EarlyStopping(monitor='val_loss', patience=5,
                        restore_best_weights=True, verbose=0)]
    hist = m.fit(
        X_train_s, y_train,
        validation_data=(X_val_s, y_val),
        epochs=50, batch_size=256, callbacks=cb, verbose=0
    )
    val_acc = max(hist.history['val_accuracy'])
    optimizer_results[opt_name] = {'val_accuracy': val_acc, 'history': hist.history}
    print(f"  Best val accuracy: {val_acc:.4f}")

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 5))
for opt_name, res in optimizer_results.items():
    ax.plot(res['history']['val_accuracy'], label=opt_name.upper())
ax.set_title('Validation Accuracy: Optimizer Comparison')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.legend()
plt.tight_layout()
plt.savefig('optimizer_comparison.png', dpi=150)
plt.show()
print("Saved: optimizer_comparison.png")

In [ ]:
# ─────────────────────────────────────────────
# CELL 10: Training curves for the best model
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(history.history['loss'],     label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

# Accuracy curve
axes[1].plot(history.history['accuracy'],     label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Training & Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig('mlp_training_curves.png', dpi=150)
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 11: Final Evaluation on Test Set
# ─────────────────────────────────────────────
y_prob = model.predict(X_test_s).flatten()   # P(positive)
y_pred = (y_prob >= 0.5).astype(int)         # threshold at 0.5

test_acc  = accuracy_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_prob)
report    = classification_report(y_test, y_pred, target_names=['negative','positive'])
cm        = confusion_matrix(y_test, y_pred)

print(f"Test Accuracy : {test_acc:.4f}")
print(f"ROC-AUC Score : {roc_auc:.4f}")
print("\nClassification Report:")
print(report)

# ── Confusion Matrix ──
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative','Positive'],
            yticklabels=['Negative','Positive'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix – Optimized MLP')
plt.tight_layout()
plt.savefig('mlp_confusion_matrix.png', dpi=150)
plt.show()

# ── ROC Curve ──
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
plt.plot([0,1],[0,1],'--', color='gray')
plt.title('ROC Curve – Optimized MLP')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.tight_layout()
plt.savefig('mlp_roc_curve.png', dpi=150)
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 12: Save Results to JSON
# ─────────────────────────────────────────────
results = {
    "model": "Optimized MLP (TF-IDF + Sentiment Features)",
    "test_accuracy": round(float(test_acc), 4),
    "roc_auc": round(float(roc_auc), 4),
    "classification_report": report,
    "optimizer_comparison": {
        k: round(v['val_accuracy'], 4) for k, v in optimizer_results.items()
    },
    "hyperparameters": {
        "hidden_layers": [256, 128, 64],
        "dropout_rate": 0.3,
        "batch_size": 256,
        "optimizer": "Adam",
        "learning_rate": 0.001,
        "batch_normalization": True,
        "early_stopping_patience": 5
    }
}

with open('results_model1_mlp.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to results_model1_mlp.json")

# Save Keras model
model.save('model1_optimized_mlp.keras')
print("Model saved to model1_optimized_mlp.keras")